## DATA CLEANSING NOTEBOOK

In [ ]:
import os

In [13]:
curr_path = !pwd
curr_path = curr_path[0] + "/"
input_file = os.path.join(os.path.dirname(curr_path), '..', 'inputs', 'inventory_raw.csv')
input_file

'/Users/jonathanperalgort/Documents/network-inventory-cleaning-and-validation/notebooks/../inputs/inventory_raw.csv'

In [37]:
import pandas as pd
import numpy as np

In [ ]:
df_raw = pd.read_csv(input_file)
df_raw.head(5)

,source_row_id,ip,hostname,fqdn,mac,owner,device_type,site,notes
0,1,192.168.010.005,HOST01,NaN,AA-BB-CC-DD-EE-FF,priya (platform) priya@corp.example.com,server,BLR Campus,db host
1,2,10.0.1.300,host-02,host-02.local,11-22-33-44-55-66,ops,NaN,HQ Bldg 1,edge gw?
2,3,10.0.1,host03,NaN,aabb.ccdd.eeff,jane@corp.example.com,switch,HQ-BUILDING-1,NaN
3,4,10.0.1.1.2,printer-01,NaN,00:11:22:33:44:55,Facilities,printer,HQ,NaN
4,5,fe80::1%eth0,iot-cam01,NaN,00:aa:bb:cc:dd:ee,sec,iot,Lab-1,camera PoE on port 3


In [23]:
df_raw.columns

Index(['source_row_id', 'ip', 'hostname', 'fqdn', 'mac', 'owner',
       'device_type', 'site', 'notes'],
      dtype='object')

### IPv4

 VALIDATION

In [78]:
def ipv4_validate_and_normalize(ip_str):
    if pd.isna(ip_str):
        return (False, None, "missing")
    s = str(ip_str).strip()
    if ':' in s:
        return (False, None, "ipv6_or_non_ipv4")
    parts = s.split(".")
    if len(parts) != 4:
        return (False, None, "wrong_octet_count")
    
    canonical_parts = []
    for p in parts:
        if p == '':
            return (False, None, "empty_octet")
        if not (p.lstrip("+").isdigit() and not p.startswith("-")):
            return (False, None, "non_numeric_or_negative")
        try:
            v = int(p, 10)
        except ValueError:
            return (False, None, "non_decimal_format")
        if v < 0 or v > 255:
            return (False, None, "octet_out_of_range")
        canonical_parts.append(str(v))
    canonical = ".".join(canonical_parts)
    
    return (True, canonical, "ok")

In [82]:
df_ipv4 = pd.DataFrame()
df_ipv4[['ip_valid', 'ip_canonical', 'ip_reason']] = df_raw['ip'].apply(
    lambda x: pd.Series(ipv4_validate_and_normalize(x))
)

In [83]:
df_ipv4

,ip_valid,ip_canonical,ip_reason
0,True,192.168.10.5,ok
1,False,None,octet_out_of_range
2,False,None,wrong_octet_count
3,False,None,wrong_octet_count
4,False,None,ipv6_or_non_ipv4
5,True,127.0.0.1,ok
6,True,169.254.10.20,ok
7,True,10.10.10.10,ok
8,False,None,non_numeric_or_negative
9,False,None,non_numeric_or_negative


 NORMALIZATION

IP_TYPE

SUBNET_CIDR